# 03c Transfer Evaluation

**Phase 3.4 - Zero-Shot Transfer Evaluation (Berlin → Leipzig)**

This notebook evaluates how well Berlin-trained models transfer to Leipzig without fine-tuning (zero-shot). Includes comprehensive transfer gap analysis, hypothesis testing, and robustness classification.

**Dependencies:**
- Berlin ML and NN champions from 03b
- Leipzig test data from 03a
- Berlin evaluation results from 03b

**Outputs:**
- `transfer_evaluation.json` - Complete transfer analysis
- Transfer figures (confusion comparison, per-genus, robustness)

---

**RUNTIME REQUIREMENTS:**
- CPU: Standard (12GB RAM sufficient)
- GPU: Not required (only inference, no training)
- Time: ~10-20 minutes

**KEY INSIGHT:** Leipzig from-scratch baseline uses Leipzig-specific scaler for fair comparison!

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.\n"
        "1. Click the key icon in the left sidebar\n"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token\n"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from GitHub
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("✅ Package installed successfully")

In [ ]:
# ============================================================
# GOOGLE DRIVE MOUNT
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted")

In [ ]:
# ============================================================
# IMPORTS & INITIALIZATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pickle
import warnings
import time
import gc

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from urban_tree_transfer.config import load_experiment_config, RANDOM_SEED
from urban_tree_transfer.experiments import (
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
)
from urban_tree_transfer.utils import (
    ExecutionLog,
    setup_plotting,
    validate_evaluation_metrics,
)

setup_plotting()
warnings.filterwarnings("ignore", category=UserWarning)

log = ExecutionLog("03c_transfer_evaluation")
start_time = time.time()

print("✅ Imports complete")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
DATA_DIR = DRIVE_DIR / "data"
INPUT_DIR = DATA_DIR / "phase_3_experiments"
OUTPUT_DIR = DATA_DIR / "phase_3_experiments"

METADATA_DIR = OUTPUT_DIR / "metadata"
MODELS_DIR = OUTPUT_DIR / "models"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "transfer_evaluation"

# Create output directories
for d in [METADATA_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Load experiment configuration
config = load_experiment_config()

print(f"Input (Phase 3):  {INPUT_DIR}")
print(f"Output:           {OUTPUT_DIR}")
print(f"Models:           {MODELS_DIR}")
print(f"Figures:          {FIGURES_DIR}")
print(f"Random seed:      {RANDOM_SEED}")
print(f"\n✅ Configuration complete")

In [ ]:
# ============================================================
# SECTION 1: Load Berlin Models & Metadata
# ============================================================

log.start_step("Load Berlin Models")

try:
    print("\n" + "=" * 70)
    print("Loading Berlin Champions")
    print("=" * 70)
    
    # Load ML champion
    ml_model_path = MODELS_DIR / "berlin_ml_champion.pkl"
    if not ml_model_path.exists():
        raise FileNotFoundError(
            f"ML champion not found at {ml_model_path}\n"
            "Run 03b_berlin_optimization.ipynb first."
        )
    
    ml_model, ml_metadata = training.load_model(ml_model_path, return_metadata=True)
    ml_name = ml_metadata["model_name"]
    
    print(f"\n✅ Loaded ML champion: {ml_name}")
    
    # Load NN champion (optional)
    nn_model_path = MODELS_DIR / "berlin_nn_champion.pt"
    if nn_model_path.exists():
        nn_model, nn_metadata = training.load_model(nn_model_path, return_metadata=True)
        nn_name = nn_metadata["model_name"]
        print(f"✅ Loaded NN champion: {nn_name}")
    else:
        nn_model = None
        nn_name = None
        print(f"ℹ️  No NN champion found - skipping NN evaluation")
    
    # Load label encoder and Berlin scaler
    with (MODELS_DIR / "label_encoder.pkl").open("rb") as f:
        label_encoder = pickle.load(f)
        label_to_idx = label_encoder["label_to_idx"]
        idx_to_label = label_encoder["idx_to_label"]
    
    with (MODELS_DIR / "berlin_scaler.pkl").open("rb") as f:
        berlin_scaler = pickle.load(f)
    
    feature_cols = ml_metadata["feature_columns"]
    class_labels = list(idx_to_label.values())
    n_classes = len(class_labels)
    
    print(f"\n  Classes: {n_classes}")
    print(f"  Features: {len(feature_cols)}")
    
    # Load Berlin evaluation results
    berlin_eval_path = METADATA_DIR / "berlin_evaluation.json"
    if not berlin_eval_path.exists():
        raise FileNotFoundError(
            f"Berlin evaluation not found at {berlin_eval_path}\n"
            "Run 03b_berlin_optimization.ipynb first."
        )
    
    berlin_eval = json.loads(berlin_eval_path.read_text())
    berlin_f1 = berlin_eval["ml_metrics"]["f1_score"]
    
    print(f"\n✅ Loaded Berlin evaluation (F1: {berlin_f1:.4f})")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 2: Load Leipzig Test Data
# ============================================================

log.start_step("Load Leipzig Data")

try:
    print("\n" + "=" * 70)
    print("Loading Leipzig Data")
    print("=" * 70)
    
    # Load Leipzig splits
    leipzig_finetune, leipzig_test = data_loading.load_leipzig_splits(INPUT_DIR)
    
    print(f"\n  Finetune: {len(leipzig_finetune):,} samples")
    print(f"  Test:     {len(leipzig_test):,} samples")
    
    # Memory optimization
    float_cols = leipzig_test.select_dtypes(include=["float64"]).columns
    if len(float_cols) > 0:
        leipzig_test[float_cols] = leipzig_test[float_cols].astype("float32")
        leipzig_finetune[float_cols] = leipzig_finetune[float_cols].astype("float32")
        print(f"  Optimized: {len(float_cols)} float64→float32")
    
    # Extract features and labels for Leipzig test
    x_test = leipzig_test[feature_cols].to_numpy()
    y_test = leipzig_test["genus_latin"].map(label_to_idx).to_numpy()
    
    # Scale with Berlin scaler (transfer learning)
    x_test_scaled = berlin_scaler.transform(x_test)
    
    print(f"\n✅ Leipzig test data prepared (scaled with Berlin scaler)")
    
    log.end_step(status="success", records=len(leipzig_test))
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 3: Zero-Shot Evaluation (ML Champion)
# ============================================================

log.start_step("Zero-Shot ML Evaluation")

try:
    print("\n" + "=" * 70)
    print(f"Zero-Shot Transfer: {ml_name.upper()}")
    print("=" * 70)
    
    # Predict with Berlin model on Leipzig data
    ml_preds = ml_model.predict(x_test_scaled)
    
    # Compute metrics with bootstrap CIs
    ml_transfer_metrics = transfer.compute_transfer_metrics(
        y_test,
        ml_preds,
        class_labels=class_labels,
        include_ci=True,
        n_bootstrap=config["metrics"]["n_bootstrap"],
    )
    
    leipzig_f1 = ml_transfer_metrics["metrics"]["f1_score"]
    
    print(f"\nZero-Shot Transfer Results:")
    print(f"  F1:       {leipzig_f1:.4f}")
    print(f"  Accuracy: {ml_transfer_metrics['metrics']['accuracy']:.4f}")
    
    if "confidence_intervals" in ml_transfer_metrics:
        f1_ci = ml_transfer_metrics["confidence_intervals"]["f1_score"]
        print(f"  F1 95% CI: [{f1_ci[0]:.4f}, {f1_ci[1]:.4f}]")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 4: Transfer Gap Analysis
# ============================================================

log.start_step("Transfer Gap Analysis")

try:
    print("\n" + "=" * 70)
    print("Transfer Gap Analysis")
    print("=" * 70)
    
    # Compute transfer gap
    gap = transfer.compute_transfer_gap(berlin_f1, leipzig_f1)
    
    print(f"\n  Berlin F1:          {berlin_f1:.4f}")
    print(f"  Leipzig F1:         {leipzig_f1:.4f}")
    print(f"  Absolute drop:      {gap.absolute_drop:.4f}")
    print(f"  Relative drop:      {gap.relative_drop:.1%}")
    print(f"  Transfer efficiency: {gap.transfer_efficiency:.1%}")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 5: Leipzig From-Scratch Baseline (M5)
# ============================================================

log.start_step("Leipzig From-Scratch Baseline")

try:
    print("\n" + "=" * 70)
    print("Leipzig From-Scratch Baseline")
    print("=" * 70)
    
    # Extract features and labels for Leipzig finetune
    x_finetune = leipzig_finetune[feature_cols].to_numpy()
    y_finetune = leipzig_finetune["genus_latin"].map(label_to_idx).to_numpy()
    
    # Fit Leipzig-specific scaler (IMPORTANT for fair comparison!)
    print(f"\nFitting Leipzig-specific scaler...")
    leipzig_scaler = StandardScaler()
    x_finetune_scaled = leipzig_scaler.fit_transform(x_finetune)
    x_test_leipzig_scaled = leipzig_scaler.transform(x_test)
    
    # Train from scratch with same hyperparameters as Berlin model
    from_scratch_params = ml_metadata.get("best_params", {})
    
    print(f"Training from scratch with Berlin hyperparameters...")
    print(f"  Training samples: {len(x_finetune_scaled):,}")
    
    from_scratch_model = models.create_model(ml_name, model_params=from_scratch_params)
    from_scratch_model.fit(x_finetune_scaled, y_finetune)
    
    # Evaluate
    from_scratch_preds = from_scratch_model.predict(x_test_leipzig_scaled)
    from_scratch_metrics = evaluation.compute_metrics(y_test, from_scratch_preds)
    from_scratch_f1 = from_scratch_metrics["f1_score"]
    
    print(f"\nFrom-Scratch Results:")
    print(f"  F1: {from_scratch_f1:.4f}")
    print(f"\nComparison:")
    print(f"  Transfer (zero-shot): {leipzig_f1:.4f}")
    print(f"  From-Scratch:         {from_scratch_f1:.4f}")
    print(f"  Difference:           {leipzig_f1 - from_scratch_f1:+.4f}")
    
    # Memory cleanup
    del x_finetune_scaled, x_test_leipzig_scaled
    gc.collect()
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 6: Feature Stability Analysis
# ============================================================

log.start_step("Feature Stability")

try:
    print("\n" + "=" * 70)
    print("Feature Stability Analysis")
    print("=" * 70)
    
    # Check if models have feature_importances_
    if hasattr(ml_model, "feature_importances_") and hasattr(
        from_scratch_model, "feature_importances_"
    ):
        # Get Berlin importance
        berlin_importance = pd.DataFrame(
            {"feature": feature_cols, "importance": ml_model.feature_importances_}
        )
        
        # Get Leipzig importance
        leipzig_importance = pd.DataFrame(
            {"feature": feature_cols, "importance": from_scratch_model.feature_importances_}
        )
        
        # Compute stability
        stability = transfer.compute_feature_stability(
            berlin_importance, leipzig_importance
        )
        
        print(f"\nFeature Stability:")
        print(f"  Spearman ρ: {stability['spearman_rho']:.3f}")
        print(f"  p-value:    {stability['p_value']:.4f}")
        
        if stability["p_value"] < 0.05:
            print(
                f"  ✅ Significant correlation between Berlin and Leipzig "
                f"feature rankings"
            )
        else:
            print(f"  ⚠️  No significant correlation in feature rankings")
    else:
        print(f"\nℹ️  Feature importance not available for {ml_name}")
        stability = None
        berlin_importance = None
        leipzig_importance = None
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 7: Per-Genus Transfer Robustness (6.3)
# ============================================================

log.start_step("Per-Genus Robustness")

try:
    print("\n" + "=" * 70)
    print("Per-Genus Transfer Robustness")
    print("=" * 70)
    
    # Get per-genus metrics
    berlin_per_genus = pd.DataFrame(berlin_eval["per_class_metrics"])
    leipzig_per_genus = ml_transfer_metrics["per_class"]
    
    # Classify robustness
    robustness = transfer.classify_transfer_robustness(
        berlin_per_genus,
        leipzig_per_genus,
        thresholds=config["transfer_evaluation"]["robustness_thresholds"],
    )
    
    # Compute ranking
    ranking = transfer.compute_transfer_robustness_ranking(robustness)
    
    # Summary
    print(f"\nTransfer Robustness Summary:")
    for category in ["robust", "medium", "poor"]:
        genera = [g for g in ranking if robustness[g]["category"] == category]
        print(f"  {category.capitalize():8s}: {len(genera):2d} genera")
    
    print(f"\nTop 5 Most Robust:")
    for genus in ranking[:5]:
        f1_drop = robustness[genus]["f1_drop"]
        category = robustness[genus]["category"]
        print(f"  {genus:15s} F1 drop: {f1_drop:+.3f} ({category})")
    
    print(f"\nTop 5 Least Robust:")
    for genus in ranking[-5:]:
        f1_drop = robustness[genus]["f1_drop"]
        category = robustness[genus]["category"]
        print(f"  {genus:15s} F1 drop: {f1_drop:+.3f} ({category})")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 8: Hypothesis Testing (6.1)
# ============================================================

log.start_step("Hypothesis Testing")

try:
    print("\n" + "=" * 70)
    print("A-Priori Hypotheses Testing")
    print("=" * 70)
    
    # Prepare data for hypotheses
    genus_data = pd.merge(
        berlin_per_genus[["genus", "f1_score", "support"]].rename(
            columns={"f1_score": "berlin_f1", "support": "berlin_n"}
        ),
        leipzig_per_genus[["genus", "f1_score"]].rename(
            columns={"f1_score": "leipzig_f1"}
        ),
        on="genus",
    )
    genus_data["transfer_gap"] = genus_data["berlin_f1"] - genus_data["leipzig_f1"]
    
    # Test each hypothesis from config
    hypotheses = config["transfer_evaluation"]["hypotheses"]
    hypothesis_results = []
    
    for hyp in hypotheses:
        print(f"\n{hyp['id']}: {hyp['description']}")
        
        result = transfer.test_hypothesis(
            hyp,
            genus_data=genus_data,
            feature_importance=berlin_importance if stability else None,
        )
        
        hypothesis_results.append(result)
        
        if "statistic" in result:
            print(f"  Statistic: {result['statistic']:.3f}")
        if "p_value" in result:
            print(f"  p-value:   {result['p_value']:.4f}")
        print(f"  Result:    {result['conclusion']}")
    
    log.end_step(status="success", records=len(hypothesis_results))
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 9: NN Champion Evaluation (if exists)
# ============================================================

log.start_step("NN Champion Evaluation")

try:
    if nn_model is None:
        print("\nℹ️  No NN champion - skipping NN evaluation")
        nn_transfer_metrics = None
        best_transfer_model = "ml"
        best_transfer_f1 = leipzig_f1
        log.end_step(status="skipped")
    else:
        print("\n" + "=" * 70)
        print(f"NN Champion Evaluation: {nn_name.upper()}")
        print("=" * 70)
        
        # Predict with NN model
        nn_preds = nn_model.predict(x_test_scaled)
        
        # Compute metrics
        nn_transfer_metrics = transfer.compute_transfer_metrics(
            y_test,
            nn_preds,
            class_labels=class_labels,
            include_ci=True,
            n_bootstrap=config["metrics"]["n_bootstrap"],
        )
        
        nn_leipzig_f1 = nn_transfer_metrics["metrics"]["f1_score"]
        
        print(f"\nNN Zero-Shot Transfer:")
        print(f"  F1:       {nn_leipzig_f1:.4f}")
        print(f"  Accuracy: {nn_transfer_metrics['metrics']['accuracy']:.4f}")
        
        print(f"\nML vs NN Comparison:")
        print(f"  ML F1: {leipzig_f1:.4f}")
        print(f"  NN F1: {nn_leipzig_f1:.4f}")
        print(f"  Difference: {nn_leipzig_f1 - leipzig_f1:+.4f}")
        
        # Select best transfer model
        if nn_leipzig_f1 > leipzig_f1:
            best_transfer_model = "nn"
            best_transfer_f1 = nn_leipzig_f1
            print(f"\n  ✅ Best transfer model: NN ({nn_name})")
        else:
            best_transfer_model = "ml"
            best_transfer_f1 = leipzig_f1
            print(f"\n  ✅ Best transfer model: ML ({ml_name})")
        
        log.end_step(status="success")

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 10: Visualizations
# ============================================================

log.start_step("Generate Visualizations")

try:
    print("\n" + "=" * 70)
    print("Generating Visualizations")
    print("=" * 70)
    
    # 1. Confusion matrix comparison
    print(f"\n1. Confusion matrix comparison...")
    berlin_cm = np.array(berlin_eval["confusion_matrix"])
    leipzig_cm = ml_transfer_metrics["confusion_matrix"]
    
    visualization.plot_confusion_comparison(
        berlin_cm,
        leipzig_cm,
        class_labels,
        output_path=FIGURES_DIR / "confusion_comparison.png",
    )
    
    # 2. Per-genus transfer
    print(f"2. Per-genus transfer plot...")
    visualization.plot_per_genus_transfer(
        berlin_per_genus,
        leipzig_per_genus,
        output_path=FIGURES_DIR / "per_genus_transfer.png",
    )
    
    # 3. Conifer vs deciduous transfer
    print(f"3. Conifer vs deciduous transfer...")
    genus_transfer_data = {
        genus: {
            "berlin_f1": next(
                (
                    r["f1_score"]
                    for r in berlin_eval["per_class_metrics"]
                    if r["genus"] == genus
                ),
                0,
            ),
            "leipzig_f1": next(
                (
                    r["f1_score"]
                    for r in ml_transfer_metrics["per_class"].to_dict("records")
                    if r["genus"] == genus
                ),
                0,
            ),
        }
        for genus in class_labels
    }
    
    visualization.plot_transfer_conifer_deciduous(
        genus_transfer_data,
        set(config["genus_groups"]["conifer"]),
        output_path=FIGURES_DIR / "transfer_conifer_deciduous.png",
    )
    
    print(f"\n✅ Visualizations saved to: {FIGURES_DIR}")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 11: Save Results
# ============================================================

log.start_step("Save Results")

try:
    # Compile transfer evaluation data
    transfer_eval_data = {
        "ml_champion": ml_name,
        "nn_champion": nn_name if nn_name else None,
        "best_transfer_model": best_transfer_model,
        "best_transfer_f1": best_transfer_f1,
        "ml_zero_shot_metrics": ml_transfer_metrics["metrics"],
        "nn_zero_shot_metrics": (
            nn_transfer_metrics["metrics"] if nn_transfer_metrics else None
        ),
        "transfer_gap": {
            "berlin_f1": berlin_f1,
            "leipzig_f1": leipzig_f1,
            "absolute_drop": gap.absolute_drop,
            "relative_drop": gap.relative_drop,
            "transfer_efficiency": gap.transfer_efficiency,
        },
        "from_scratch_f1": from_scratch_f1,
        "feature_stability": stability if stability else None,
        "per_genus_robustness": robustness,
        "robustness_ranking": ranking,
        "hypothesis_tests": hypothesis_results,
        "per_class_metrics": ml_transfer_metrics["per_class"].to_dict(orient="records"),
        "confusion_matrix": ml_transfer_metrics["confusion_matrix"].tolist(),
    }
    
    # Save transfer evaluation
    eval_path = METADATA_DIR / "transfer_evaluation.json"
    eval_path.write_text(json.dumps(transfer_eval_data, indent=2))
    validate_evaluation_metrics(eval_path)
    
    print(f"\n✅ Saved: {eval_path.name}")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 12: Save Execution Log
# ============================================================

log.start_step("Save Execution Log")

try:
    elapsed = time.time() - start_time
    
    # Create execution summary
    execution_summary = {
        "notebook": "03c_transfer_evaluation",
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "runtime_seconds": elapsed,
        "runtime_minutes": elapsed / 60,
        "champions": {
            "ml": ml_name,
            "nn": nn_name if nn_name else None,
        },
        "best_transfer_model": best_transfer_model,
        "transfer_gap": {
            "berlin_f1": berlin_f1,
            "leipzig_f1": leipzig_f1,
            "absolute_drop": gap.absolute_drop,
            "relative_drop": gap.relative_drop,
        },
        "robustness_summary": {
            "robust": len([g for g in ranking if robustness[g]["category"] == "robust"]),
            "medium": len([g for g in ranking if robustness[g]["category"] == "medium"]),
            "poor": len([g for g in ranking if robustness[g]["category"] == "poor"]),
        },
        "hypothesis_tests": len(hypothesis_results),
    }
    
    # Save execution log
    log.summary()
    log_path = LOGS_DIR / f"{log.notebook}_execution.json"
    log.save(log_path)
    
    # Save summary
    summary_path = METADATA_DIR / "03c_summary.json"
    summary_path.write_text(json.dumps(execution_summary, indent=2))
    
    print(f"\n✅ Execution log saved: {log_path}")
    print(f"✅ Summary saved: {summary_path}")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("NOTEBOOK COMPLETE: 03c Transfer Evaluation")
print("=" * 70)

print(f"\nRuntime: {elapsed/60:.1f} minutes")

print(f"\nTransfer Performance:")
print(f"  Berlin F1:          {berlin_f1:.4f}")
print(f"  Leipzig F1:         {leipzig_f1:.4f}")
print(f"  Transfer gap:       {gap.relative_drop:.1%}")
print(f"  Transfer efficiency: {gap.transfer_efficiency:.1%}")

print(f"\nBaseline Comparison:")
print(f"  Transfer (zero-shot): {leipzig_f1:.4f}")
print(f"  From-Scratch:         {from_scratch_f1:.4f}")
print(f"  Difference:           {leipzig_f1 - from_scratch_f1:+.4f}")

if nn_model:
    print(f"\nML vs NN:")
    print(f"  ML F1: {leipzig_f1:.4f}")
    print(f"  NN F1: {nn_transfer_metrics['metrics']['f1_score']:.4f}")
    print(f"  Best: {best_transfer_model.upper()}")

print(f"\nRobustness Summary:")
for category in ["robust", "medium", "poor"]:
    count = len([g for g in ranking if robustness[g]["category"] == category])
    print(f"  {category.capitalize()}: {count} genera")

print(f"\nOutputs:")
print(f"  Metadata: {eval_path.name}")
print(f"  Summary:  03c_summary.json")
print(f"  Figures:  {FIGURES_DIR}")
print(f"  Logs:     {log_path}")

print(f"\nNext Steps:")
print(f"  1. Review transfer gap analysis")
print(f"  2. Inspect per-genus robustness classification")
print(f"  3. Analyze hypothesis test results")
print(f"  4. Run 03d_finetuning.ipynb (sample efficiency)")
print("=" * 70)